# DA6401 Report - Section 2.6 Loss Function Comparison

Controlled comparison between:
- Mean Squared Error (MSE)
- Cross-Entropy

All other settings are held constant.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import wandb

import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from train import train_and_evaluate

print("Project root:", ROOT)


In [ ]:
RUN_EXPERIMENT = False  # Set True to launch W&B runs.

PROJECT = "da6401_assignment_1"
ENTITY = None
RUN_GROUP = "report_v1_2_6_loss_comparison"

common = dict(
    dataset="mnist",
    epochs=15,
    batch_size=64,
    optimizer="rmsprop",
    learning_rate=0.001,
    weight_decay=0.0,
    num_layers=3,
    hidden_size=[128, 128, 128],
    activation="relu",
    weight_init="xavier",
    wandb_project=PROJECT,
    wandb_entity=ENTITY,
    wandb_mode="online",
    seed=42,
)

configs = [
    {"loss": "mean_squared_error", "label": "mse"},
    {"loss": "cross_entropy", "label": "cross_entropy"},
]

out_dir = ROOT / "src" / "tmp"
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
results = []

if RUN_EXPERIMENT:
    for cfg in configs:
        args = SimpleNamespace(
            loss=cfg["loss"],
            model_path=str(out_dir / f"model_2_6_{cfg['label']}.npy"),
            config_path=str(out_dir / f"config_2_6_{cfg['label']}.json"),
            **common,
        )

        run = wandb.init(
            project=PROJECT,
            entity=ENTITY,
            name=f"report_2_6_{cfg['label']}",
            group=RUN_GROUP,
            tags=["report_v1", "report_section_2.6", "loss_comparison", "mnist", cfg["loss"]],
            config=vars(args),
        )

        result = train_and_evaluate(args, wandb_run_override=run)
        hist = result["history"]

        train_loss_curve = [float(ep["train"]["loss"]) for ep in hist]
        val_loss_curve = [float(ep["val"]["loss"]) for ep in hist]
        train_acc_curve = [float(ep["train"]["accuracy"]) for ep in hist]
        val_acc_curve = [float(ep["val"]["accuracy"]) for ep in hist]

        final_val_acc = val_acc_curve[-1]
        threshold = 0.95 * final_val_acc
        conv_epoch = None
        for i, v in enumerate(val_acc_curve, start=1):
            if v >= threshold:
                conv_epoch = i
                break

        rec = {
            "label": cfg["label"],
            "loss": cfg["loss"],
            "run_id": run.id,
            "run_name": run.name,
            "train_loss_curve": train_loss_curve,
            "val_loss_curve": val_loss_curve,
            "train_acc_curve": train_acc_curve,
            "val_acc_curve": val_acc_curve,
            "final_val_acc": final_val_acc,
            "final_val_f1": float(result["final_metrics"]["val"]["f1"]),
            "final_test_acc": float(result["final_metrics"]["test"]["accuracy"]),
            "final_test_f1": float(result["final_metrics"]["test"]["f1"]),
            "epoch5_train_loss": train_loss_curve[4],
            "epoch5_val_loss": val_loss_curve[4],
            "epoch5_val_acc": val_acc_curve[4],
            "convergence_epoch_95pct_final_val_acc": conv_epoch,
        }
        results.append(rec)

        run.summary["analysis/epoch5_train_loss"] = rec["epoch5_train_loss"]
        run.summary["analysis/epoch5_val_loss"] = rec["epoch5_val_loss"]
        run.summary["analysis/epoch5_val_acc"] = rec["epoch5_val_acc"]
        run.summary["analysis/convergence_epoch_95pct_final_val_acc"] = rec["convergence_epoch_95pct_final_val_acc"]
        run.finish()

    plt.figure(figsize=(9, 5))
    for rec in results:
        plt.plot(range(1, len(rec["train_loss_curve"]) + 1), rec["train_loss_curve"], marker="o", linewidth=2, label=rec["label"])
    plt.xlabel("Epoch")
    plt.ylabel("Train Loss")
    plt.title("2.6 Train Loss Comparison (Same Architecture/LR)")
    plt.grid(alpha=0.3)
    plt.legend()
    train_loss_plot = out_dir / "report_2_6_train_loss_comparison.png"
    plt.tight_layout(); plt.savefig(train_loss_plot, dpi=180); plt.close()

    plt.figure(figsize=(9, 5))
    for rec in results:
        plt.plot(range(1, len(rec["val_loss_curve"]) + 1), rec["val_loss_curve"], marker="o", linewidth=2, label=rec["label"])
    plt.xlabel("Epoch")
    plt.ylabel("Validation Loss")
    plt.title("2.6 Validation Loss Comparison (Same Architecture/LR)")
    plt.grid(alpha=0.3)
    plt.legend()
    val_loss_plot = out_dir / "report_2_6_val_loss_comparison.png"
    plt.tight_layout(); plt.savefig(val_loss_plot, dpi=180); plt.close()

    plt.figure(figsize=(9, 5))
    for rec in results:
        plt.plot(range(1, len(rec["val_acc_curve"]) + 1), rec["val_acc_curve"], marker="o", linewidth=2, label=rec["label"])
    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy")
    plt.title("2.6 Validation Accuracy Comparison (Same Architecture/LR)")
    plt.grid(alpha=0.3)
    plt.legend()
    val_acc_plot = out_dir / "report_2_6_val_accuracy_comparison.png"
    plt.tight_layout(); plt.savefig(val_acc_plot, dpi=180); plt.close()

    summary_run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name="report_2_6_summary",
        group=RUN_GROUP,
        tags=["report_v1", "report_section_2.6", "loss_comparison", "summary"],
    )

    table = wandb.Table(columns=[
        "label", "loss", "run_id",
        "epoch5_train_loss", "epoch5_val_loss", "epoch5_val_acc",
        "final_val_acc", "final_val_f1", "final_test_acc", "final_test_f1",
        "convergence_epoch_95pct_final_val_acc"
    ])
    for rec in results:
        table.add_data(
            rec["label"], rec["loss"], rec["run_id"],
            rec["epoch5_train_loss"], rec["epoch5_val_loss"], rec["epoch5_val_acc"],
            rec["final_val_acc"], rec["final_val_f1"], rec["final_test_acc"], rec["final_test_f1"],
            rec["convergence_epoch_95pct_final_val_acc"],
        )

    summary_run.log({
        "analysis/loss_comparison_table": table,
        "analysis/train_loss_comparison_plot": wandb.Image(str(train_loss_plot)),
        "analysis/val_loss_comparison_plot": wandb.Image(str(val_loss_plot)),
        "analysis/val_accuracy_comparison_plot": wandb.Image(str(val_acc_plot)),
    })
    summary_run.finish()

    payload = {
        "setup": common,
        "results": results,
        "summary_run_name": "report_2_6_summary",
        "plots": {
            "train_loss": str(train_loss_plot),
            "val_loss": str(val_loss_plot),
            "val_accuracy": str(val_acc_plot),
        },
    }
    with open(out_dir / "report_2_6_loss_comparison.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print("Completed 2.6 runs and summary")
else:
    print("RUN_EXPERIMENT is False. No runs were launched.")


In [ ]:
summary_path = out_dir / "report_2_6_loss_comparison.json"
if summary_path.exists():
    data = json.loads(summary_path.read_text())
    for rec in data["results"]:
        print(rec["label"], rec["run_id"],
              "e5_val_loss", round(rec["epoch5_val_loss"], 6),
              "e5_val_acc", round(rec["epoch5_val_acc"], 6),
              "final_val_acc", round(rec["final_val_acc"], 6))
else:
    print("No summary file yet. Run experiment cells first.")
